# Load deps

In [ ]:
# !pip install -q evaluate rouge_score nltk

In [ ]:
import os, torch, evaluate
from datasets import load_dataset, load_dataset_builder
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainer
import matplotlib.pyplot as plt
from datasets import concatenate_datasets, DatasetDict
import nltk
from nltk.tokenize import sent_tokenize
import pandas as pd
import numpy as np

# HF Login

In [ ]:
secret_label = "HF_TOKEN"
os.environ[secret_label] = UserSecretsClient().get_secret(secret_label)
!hf auth whoami

# Config

- We’ll use the [Multilingual Amazon Reviews Corpus](https://huggingface.co/datasets/amazon_reviews_multi) to create our bilingual summarizer.
- This corpus consists of Amazon product reviews in six languages and is typically used to benchmark multilingual classifiers.
- However, since each review is accompanied by a short title, we can use the titles as the target summaries for our model to learn from!

In [ ]:
hf_user_name = "amin-oj"
model_checkpoint = "google/mt5-small"
dataset_dict = dict(path = "buruzaemon/amazon_reviews_multi")
max_input_length = 256
max_target_length = 30
push_to_hub=True
train_bs = 16
eval_bs = 16
num_train_epochs = 4
task = "summarization"
ckpt_name = f"{model_checkpoint.split("/")[-1]}-finetuned-{task}-{dataset_dict['path'].split("/")[-1]}"
print(ckpt_name)

# Load data

In [ ]:
lang = "en"
url = f"https://huggingface.co/datasets/{dataset_dict["path"]}/resolve/main/{lang}"
data_files = {
    split: f"{url}/{split}.jsonl.gz" for split in ["train", "test", "validation"]
}
english_dataset = load_dataset("json", data_files=data_files)

In [ ]:
lang = "es"
url = f"https://huggingface.co/datasets/{dataset_dict["path"]}/resolve/main/{lang}"
data_files = {
    split: f"{url}/{split}.jsonl.gz" for split in ["train", "test", "validation"]
}
spanish_dataset = load_dataset("json", data_files=data_files)

In [ ]:
def show_samples(dataset, num_samples=3, seed=42):
    sample = dataset["train"].shuffle(seed=seed).select(range(num_samples))
    for example in sample:
        print(f"\n'>> Title: {example['review_title']}'")
        print(f"'>> Review: {example['review_body']}'")


# show_samples(english_dataset)

# Prep data

-  Training a summarization model on all 400,000 reviews would take far too long on a single GPU,
-  so instead we’ll focus on generating summaries for a single domain of products.

In [ ]:
english_dataset.set_format("pandas")
english_df = english_dataset["train"][:]
english_df["product_category"].value_counts()[:20]

In [ ]:
def filter_books(examples):
    return [
        (
            (pcat == "book")
            # or (pcat == "digital_ebook_purchase")
        )
        for pcat in examples["product_category"]
    ]

english_dataset.reset_format()
spanish_books = spanish_dataset.filter(filter_books, batched=True)
english_books = english_dataset.filter(filter_books, batched=True)
# show_samples(english_books)

In [ ]:
books_dataset = DatasetDict()

# stack two `Dataset` objects on top of each other to create a bilingual dataset
for split in english_books.keys():
    books_dataset[split] = concatenate_datasets(
        [english_books[split], spanish_books[split]]
    )
    #  shuffle the result to ensure our model doesn’t overfit to a single language
    books_dataset[split] = books_dataset[split].shuffle(seed=42)

# show_samples(books_dataset)

## Word count distribution

In [ ]:
book_df = books_dataset["train"].to_pandas()
print(book_df.shape)
fig, axs = plt.subplots(1,2, figsize = (10,4))
book_df["review_title"].map(lambda x: len(x.split())).plot(kind="hist", bins=30, ax=axs[0], title="review title");
book_df["review_body"].map(lambda x: len(x.split())).plot(kind="hist", bins=30, ax=axs[1], title="review body");

- short reference summaries in the data can bias the model to only output one or two words in the generated summaries.
- we’ll filter out the examples with very short titles so that our model can produce more interesting summaries.

In [ ]:
books_dataset = books_dataset.filter(lambda x: map(lambda x: len(x.split()) > 2, x["review_title"]), batched=True)

## Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
inputs = tokenizer("I loved reading the Hunger Games!")
print(inputs)
print("=================")
print(tokenizer.convert_ids_to_tokens(inputs.input_ids))

- The special Unicode character `▁` and end-of-sequence token `</s>` indicate that we’re dealing with the SentencePiece tokenizer, which is based on the Unigram segmentation algorithm
- Unigram is especially useful for multilingual corpora since it allows SentencePiece to be agnostic about accents, punctuation, and the fact that many languages, like Japanese, do not have whitespace characters.
- To tokenize our corpus, we have to deal with a subtlety associated with summarization
    - because our labels are also text, it is possible that they exceed the model’s maximum context size.
- This means we need to apply truncation to both the reviews and their titles to ensure we don’t pass excessively long inputs to our model.

In [ ]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["review_body"],
        max_length=max_input_length,
        truncation=True,
    )
    labels = tokenizer(
        # input as text_target
        text_target = examples["review_title"],
        max_length=max_target_length,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_datasets = books_dataset.map(preprocess_function, batched=True)

# Metrics

- For summarization, one of the most commonly used metrics is the [ROUGE score](https://en.wikipedia.org/wiki/ROUGE_(metric)) (short for Recall-Oriented Understudy for Gisting Evaluation).
- The basic idea behind this metric is to compare a generated summary against a set of reference summaries that are typically created by humans.
- assume the following example:
```
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"
```
- One way to compare them could be to count the number of overlapping words, which in this case would be 6.
- However, this is a bit crude, so instead, ROUGE is based on computing the precision and recall scores for the overlap.
- For ROUGE, recall measures how much of the reference summary is captured by the generated one according to the following formula:
${Recall} = \frac{\mathrm{Number\,of\,overlapping\, words}}{\mathrm{Total\, number\, of\, words\, in\, reference\, summary}}$
- For our simple example above, this formula gives a perfect recall of 6/6 = 1
-  This may sound great, but imagine if our generated summary had been “I really really loved reading the Hunger Games all night”.
- To penalize **verbose** scenarios we also compute the precision, which in the ROUGE context measures how much of the generated summary was relevant
${Precision} = \frac{\mathrm{Number\,of\,overlapping\, words}}{\mathrm{Total\, number\, of\, words\, in\, generated\, summary}}$
- Applying this to our verbose summary gives a precision of 6/10 = 0.6, which is considerably worse than the precision of 6/7 = 0.86 obtained by our shorter one.
- both precision and recall are usually computed, and then the F1-score (the harmonic mean of precision and recall) is reported.

In [ ]:
rouge_score = evaluate.load("rouge")
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"
scores = rouge_score.compute(
    predictions=[generated_summary], references=[reference_summary]
)
print(scores)

-  The rouge1 variant is the overlap of unigrams
    -  this is just a fancy way of saying the overlap of words and is exactly the metric(f1 score) we’ve discussed above.
-  rouge2 measures the overlap between bigrams (think the overlap of pairs of words)
-  while rougeL and rougeLsum measure the longest matching sequences of words by looking for the longest common substrings in the generated and reference summaries.
    -  The “sum” in rougeLsum refers to the fact that this metric is computed over a whole summary, while rougeL is computed as the average over individual sentences.

## Creating a strong baseline

- A common baseline for text summarization is to simply take the first three sentences of an article, often called the _lead-3_ baseline.
- We could use full stops to track the sentence boundaries, but this will fail on acronyms like “U.S.” or “U.N.”
- so instead we’ll use the `nltk` library, which includes a better algorithm to handle these cases.

In [ ]:
nltk.download("punkt")

def three_sentence_summary(text):
    return "\n".join(sent_tokenize(text)[:3])


print(three_sentence_summary(books_dataset["train"][1]["review_body"]))

In [ ]:
def evaluate_baseline(dataset, metric):
    summaries = [three_sentence_summary(text) for text in dataset["review_body"]]
    return metric.compute(predictions=summaries, references=dataset["review_title"])

In [ ]:
score = evaluate_baseline(books_dataset["validation"], rouge_score)
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = dict((rn, round(score[rn] * 100, 2)) for rn in rouge_names)
print(rouge_dict)

- We can see that the `rouge2` score is significantly lower than the rest
- this likely reflects the fact that review titles are typically concise and so the lead-3 baseline is too verbose.
- Now that we have a good baseline to work from, let’s turn our attention toward fine-tuning mT5!

# Fine-tuning mT5

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

In [ ]:
logging_steps = len(tokenized_datasets["train"]) // train_bs
args = Seq2SeqTrainingArguments(
    output_dir=ckpt_name,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    num_train_epochs=num_train_epochs,
    push_to_hub=push_to_hub,
    logging_steps=logging_steps,
    eval_strategy="epoch",
    learning_rate=5.6e-5,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    # Decode generated summaries into text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # Decode reference summaries into text
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # ROUGE expects a newline after each sentence
    decoded_preds = ["\n".join(sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(sent_tokenize(label.strip())) for label in decoded_labels]
    # Compute ROUGE scores
    result = rouge_score.compute(
        predictions=decoded_preds, references=decoded_labels, use_stemmer=True
    )
    # Extract the median scores
    result = {key: value * 100 for key, value in result.items()}
    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# we need to remove the columns with strings
# because the collator won’t know how to pad these elements
tokenized_datasets = tokenized_datasets.remove_columns(
    books_dataset["train"].column_names
)

In [ ]:
features = [tokenized_datasets["train"][i] for i in range(2)]
print(data_collator(features))

In [ ]:
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
trainer.push_to_hub(commit_message="Training complete", tags=task)

# Using your fine-tuned model

In [ ]:
# model = AutoModelForSeq2SeqLM.from_pretrained(f"{hf_user_name}/{ckpt_name}")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

def print_summary(idx):
    review = books_dataset["test"][idx]["review_body"]
    title = books_dataset["test"][idx]["review_title"]
    # generation
    inputs = tokenizer(review, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs)
    predictions = outputs[0]
    # predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    summary = tokenizer.decode(predictions, skip_special_tokens=True)
    print(f"'>>> Review: {review}'")
    print(f"\n'>>> Title: {title}'")
    print(f"\n'>>> Summary: {summary}'")

print_summary(12)